# Week 3 Community Contribution - AdnanGobeljic

This notebook implements a Week 3 style **Synthetic Dataset Generator**.

It covers:
- synthetic data generation with LLMs
- model switching (OpenAI + Ollama)
- Gradio UI for interactive usage
- clean export as JSON or CSV
- robust parsing and graceful error handling

In [ ]:
# imports

import os
import re
import csv
import json
from io import StringIO
from pathlib import Path
from datetime import datetime
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [ ]:
# setup clients

load_dotenv(override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = None
if openai_api_key and openai_api_key.strip() == openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key)

ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def _is_ollama_running():
    try:
        response = requests.get("http://localhost:11434", timeout=2)
        return response.status_code == 200
    except Exception:
        return False

OLLAMA_AVAILABLE = _is_ollama_running()


In [ ]:
# prompts and presets

DATASET_PRESETS = {
    "Q&A": "question and answer pairs",
    "Classification": "short text with class labels",
    "Customer Support": "customer support tickets",
    "E-commerce Products": "product catalog records",
    "Healthcare": "synthetic healthcare observations",
}

SYSTEM_PROMPT = """
You generate synthetic datasets for ML and LLM experiments.
Always output ONLY valid JSON as an array of objects.
Do not include markdown, code fences, comments, or explanations.
Never include real personal data. Use fictional values only.
"""


In [ ]:
def resolve_client(model_choice):
    if model_choice == "GPT (OpenAI)":
        if openai_client is None:
            return None, None, "OpenAI is not configured. Add OPENAI_API_KEY to .env."
        return openai_client, OPENAI_MODEL, None

    if model_choice == "Llama (Ollama)":
        if not OLLAMA_AVAILABLE:
            return None, None, "Ollama is not running. Start it with: ollama serve"
        return ollama_client, OLLAMA_MODEL, None

    return None, None, "Unknown model selected."


def strip_code_fences(text):
    cleaned = (text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    return cleaned.strip()


def parse_json_records(raw_text):
    cleaned = strip_code_fences(raw_text)

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return parsed, None
        return None, "Model output is valid JSON but not an array."
    except Exception:
        match = re.search(r"\\[.*\\]", cleaned, flags=re.DOTALL)
        if not match:
            return None, "Could not find a JSON array in model output."
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return parsed, None
            return None, "Extracted JSON content is not an array."
        except Exception:
            return None, "Failed to parse JSON array from model output."


def records_to_csv(records):
    if not records:
        return ""

    all_fields = []
    seen = set()
    for row in records:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                all_fields.append(key)

    buffer = StringIO()
    writer = csv.DictWriter(buffer, fieldnames=all_fields)
    writer.writeheader()
    for row in records:
        writer.writerow({k: row.get(k, "") for k in all_fields})
    return buffer.getvalue()


def resolve_output_dir():
    preferred = Path("week3/community-contributions/AdnanGobeljic")
    if preferred.exists():
        preferred.mkdir(parents=True, exist_ok=True)
        return preferred

    fallback = Path(".")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


OUTPUT_DIR = resolve_output_dir()


In [ ]:
def build_user_prompt(preset, topic, schema_text, rows):
    base_shape = DATASET_PRESETS.get(preset, "structured records")
    schema_text = (schema_text or "").strip()

    prompt = f"""
Generate exactly {int(rows)} synthetic records.

Dataset style: {preset} ({base_shape})
Domain/topic: {topic.strip()}
"""

    if schema_text:
        prompt += f"Requested fields/schema:\n{schema_text}\n"
    else:
        prompt += "If schema is not provided, infer a practical schema for the chosen dataset style.\n"

    prompt += """
Output requirements:
- Return only a JSON array of objects.
- Use realistic, diverse synthetic values.
- Keep keys in snake_case.
- Avoid personally identifiable real data.
"""

    return prompt.strip()


def generate_dataset(preset, topic, schema_text, rows, model_choice, output_format):
    if not topic or not topic.strip():
        return "Please provide a topic.", "", None

    client, model_id, error = resolve_client(model_choice)
    if error:
        return error, "", None

    rows = max(1, min(100, int(rows)))
    user_prompt = build_user_prompt(preset, topic, schema_text, rows)

    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
    except Exception as exc:
        return f"Model call failed: {exc}", "", None

    raw = response.choices[0].message.content or ""
    records, parse_error = parse_json_records(raw)
    if parse_error:
        return f"Output parsing failed: {parse_error}", strip_code_fences(raw)[:3000], None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if output_format == "CSV":
        output_text = records_to_csv(records)
        file_path = OUTPUT_DIR / f"synthetic_dataset_{timestamp}.csv"
    else:
        output_text = json.dumps(records, indent=2)
        file_path = OUTPUT_DIR / f"synthetic_dataset_{timestamp}.json"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(output_text)

    status = f"Generated {len(records)} records using {model_choice}."
    return status, output_text[:5000], str(file_path)


In [ ]:
with gr.Blocks(title="Week 3 Synthetic Dataset Generator") as demo:
    gr.Markdown("# Week 3 - Synthetic Dataset Generator (AdnanGobeljic)")
    gr.Markdown("Generate synthetic records with GPT or local Llama, then export JSON or CSV.")

    with gr.Row():
        preset = gr.Dropdown(
            choices=list(DATASET_PRESETS.keys()),
            value="Q&A",
            label="Dataset style",
        )
        model_choice = gr.Dropdown(
            choices=["GPT (OpenAI)", "Llama (Ollama)"],
            value="GPT (OpenAI)",
            label="Model",
        )

    topic = gr.Textbox(
        label="Topic/domain",
        placeholder="e.g. Python debugging, e-commerce returns, healthcare triage",
        lines=2,
    )

    schema_text = gr.Textbox(
        label="Optional schema/fields",
        placeholder="e.g. id, customer_issue, priority, resolution, sentiment",
        lines=3,
    )

    with gr.Row():
        rows = gr.Slider(1, 100, value=10, step=1, label="Number of records")
        output_format = gr.Radio(["JSON", "CSV"], value="JSON", label="Export format")

    generate_btn = gr.Button("Generate Dataset", variant="primary")

    status = gr.Textbox(label="Status", interactive=False)
    preview = gr.Textbox(label="Preview", lines=14)
    download_file = gr.File(label="Download")

    generate_btn.click(
        fn=generate_dataset,
        inputs=[preset, topic, schema_text, rows, model_choice, output_format],
        outputs=[status, preview, download_file],
    )

    gr.Examples(
        examples=[
            [
                "Q&A",
                "LLM engineering interview prep",
                "question, answer, difficulty, topic",
                8,
                "GPT (OpenAI)",
                "JSON",
            ],
            [
                "Customer Support",
                "subscription billing issues",
                "ticket_id, issue_summary, priority, status, resolution_time_hours",
                12,
                "Llama (Ollama)",
                "CSV",
            ],
        ],
        inputs=[preset, topic, schema_text, rows, model_choice, output_format],
        label="Try examples",
    )


In [ ]:
demo.launch()

## Notes

- If GPT is unavailable, add `OPENAI_API_KEY` to `.env`.
- If Llama is unavailable, run `ollama serve` and pull a model like `llama3.2`.
- The app always asks the model for JSON and then converts to CSV locally for better consistency.